<a href="https://colab.research.google.com/github/SAILResearch/AI_Teammates_in_SE3/blob/main/analysis/load_AIDev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loading AIDev without running out of memory

A guided tour of the memory-safe ways to read the [AIDev](https://huggingface.co/datasets/hao-li/AIDev) dataset (v5: 7.6M
agent-authored pull requests across 961k repositories). Each numbered section is an
independent, copy-pasteable example.

### The three rules

AIDev v5 is big: `all_pull_request.parquet` is 6.7 GB on disk (7.6M rows) and
`pr_commit_details.parquet` is 12.4 GB (12.1M rows). A plain `pd.read_parquet(path)` on
either one needs tens of GB of RAM. Three rules keep memory flat:

1. **Select columns.** `body`, `title` and `patch` hold most of the bytes; skip them unless you need them.
2. **Stream batches.** `pyarrow.parquet.ParquetFile(path).iter_batches(...)` yields a slice of rows at a time; reduce each slice and keep only the aggregate.
3. **Filter while reading.** `pq.read_table(path, columns=[...], filters=[...])` pushes predicates into the reader, so unwanted rows never reach pandas.

The same code is available as plain scripts in the dataset repository
([`scripts/`](https://huggingface.co/datasets/hao-li/AIDev/tree/v5/scripts)); the
[`scripts/README.md`](https://huggingface.co/datasets/hao-li/AIDev/blob/v5/scripts/README.md)
there has a table cheat sheet with join keys and measured run times.


## Setup

Everything below needs only `pandas`, `pyarrow`, `matplotlib` and `huggingface_hub`
(all preinstalled on Colab; otherwise `pip install pandas pyarrow matplotlib huggingface_hub`).


In [1]:
import os
import time
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

HF_REPO = "hao-li/AIDev"
HF_REVISION = "v5"   # pin a release (the tags are listed on the dataset card); "main" = latest

# Tables are downloaded once into the Hugging Face cache (~/.cache/huggingface), so
# re-running the notebook does not download again. If you already have the parquet
# files (e.g. a clone of the dataset repo), point AIDEV_DATA_DIR at that folder.
DATA_DIR = os.environ.get("AIDEV_DATA_DIR")


def table_path(name: str) -> Path:
    """Local path of `<name>.parquet`, downloading it on first use."""
    if DATA_DIR:
        return Path(DATA_DIR) / f"{name}.parquet"
    return Path(hf_hub_download(HF_REPO, f"{name}.parquet", repo_type="dataset",
                                revision=HF_REVISION))


OUT_DIR = Path("output")   # tables and figures written by this notebook
OUT_DIR.mkdir(exist_ok=True)
SCOPE = "all"   # "pop" = AIDev-pop (repositories with > 100 stars, 362 MB)
                 # "all" = every repository (all 7.6M PRs, a one-off 6.7 GB download)
PR_TABLE = "all_pull_request" if SCOPE == "all" else "pull_request"
REPO_TABLE = "all_repository" if SCOPE == "all" else "repository"

# Column sets reused throughout. Note what is NOT here: title, body, patch.
LIGHT_PR_COLS = ["id", "number", "agent", "user", "state",
                 "created_at", "closed_at", "merged_at", "repo_url"]
BATCH_ROWS = 65_536   # rows per streamed batch; smaller = lower peak memory, same speed

ALL_TABLES = [
    # full scope (every repository)
    "all_pull_request", "all_repository", "all_user",
    # AIDev-pop (> 100 stars) and its per-PR sub-resources
    "pull_request", "repository", "pr_comments", "pr_reviews",
    "pr_review_comments", "pr_commits", "pr_commit_details", "pr_timeline",
    "issue", "related_issue",
]

## 1. Know your tables before loading anything

Parquet keeps row counts and per-column sizes in its footer, and the Hub serves range
requests, so `ParquetFile` fetches a few KB per table instead of the file. This is how you
find out *which* columns are heavy: `body` is ~75% of `all_pull_request`, `patch` is ~90%
of `pr_commit_details`.

In [2]:
from huggingface_hub import HfFileSystem

fs = HfFileSystem()
rows = []
for name in ALL_TABLES:
    with fs.open(f"datasets/{HF_REPO}@{HF_REVISION}/{name}.parquet", "rb") as f:
        md = pq.ParquetFile(f).metadata          # footer only, nothing else is read
    size_by_col: dict[str, int] = {}
    for rg in range(md.num_row_groups):
        for ci in range(md.num_columns):
            col = md.row_group(rg).column(ci)
            size_by_col[col.path_in_schema] = (
                size_by_col.get(col.path_in_schema, 0) + col.total_compressed_size)
    total = sum(size_by_col.values())
    heavy = sorted(size_by_col.items(), key=lambda kv: -kv[1])[:2]
    rows.append({"table": name, "rows": f"{md.num_rows:,}", "GB on disk": f"{total / 1e9:.2f}",
                 "heaviest columns": ", ".join(f"{c} {s / total:.0%}" for c, s in heavy)})
pd.DataFrame(rows).set_index("table")

,rows,GB on disk,heaviest columns
table,,,
all_pull_request,"7,685,281",6.73,"body 87%, title 4%"
all_repository,"957,209",0.06,"url 42%, full_name 41%"
all_user,"399,962",0.01,"login 38%, created_at 33%"
pull_request,"361,296",0.36,"body 90%, title 4%"
repository,"25,402",0.00,"url 38%, full_name 37%"
pr_comments,"676,098",0.69,"body 98%, created_at 1%"
pr_reviews,"541,831",0.21,"body 93%, submitted_at 2%"
pr_review_comments,"291,248",0.33,"diff_hunk 66%, body 24%"
pr_commits,"1,718,829",0.34,"message 77%, sha 20%"


## 2. Small tables: load them fully, this is fine

For anything under ~100 MB an `hf://` URL in `pd.read_parquet` is all you need
(`@v5` pins the release; it is part of the path).

In [3]:
repos = pd.read_parquet(f"hf://datasets/{HF_REPO}@{HF_REVISION}/all_repository.parquet")
users = pd.read_parquet(f"hf://datasets/{HF_REPO}@{HF_REVISION}/all_user.parquet")
pop_repos = pd.read_parquet(f"hf://datasets/{HF_REPO}@{HF_REVISION}/repository.parquet")
print(f"all_repository: {len(repos):,} rows | all_user: {len(users):,} rows | "
      f"repository (>100 stars): {len(pop_repos):,} rows")
repos[["full_name", "language", "stars", "forks", "license"]].head()

all_repository: 957,209 rows | all_user: 399,962 rows | repository (>100 stars): 25,402 rows


,full_name,language,stars,forks,license
0,JAYPALBAGHEL/newyear.git.io,None,0.0,0.0,None
1,dlt-hub/dlt,Python,5195.0,490.0,Apache-2.0
2,mzeidhassan/scraper,JavaScript,0.0,0.0,BSD-3-Clause
3,lextiz/SolidHA,Python,0.0,0.0,Apache-2.0
4,researchwithkeval/TRY_one,None,0.0,0.0,None


## 3. Big tables: ALWAYS select columns (and prefer the Arrow string backend)

The PR table is downloaded once (`table_path`) and read with 9 light columns. Measured on
the full 7.6M-row table: `dtype_backend="pyarrow"` gives 1.4 GB in RAM in ~2 s; the default
object-string backend needs 3.7 GB and ~40 s, because every string becomes a separate
Python object. Everything you do with pandas still works on the Arrow-backed frame.

In [4]:
t0 = time.time()
prs = pd.read_parquet(table_path(PR_TABLE), columns=LIGHT_PR_COLS, dtype_backend="pyarrow")
mb = prs.memory_usage(deep=True).sum() / 1e6
print(f"{len(prs):,} PRs x {len(LIGHT_PR_COLS)} columns = {mb:,.0f} MB in RAM "
      f"({time.time() - t0:.1f} s incl. download on first run)")
prs.groupby("agent").size().sort_values(ascending=False).rename("# PR").to_frame()

7,685,281 PRs x 9 columns = 1,369 MB in RAM (2.0 s incl. download on first run)


,# PR
agent,
OpenAI_Codex,3844602
Copilot,1581924
Claude_Code,1567100
Cursor,317453
Google_Jules,296411
Devin,77791


In [5]:
del prs   # free it before the next section

## 4. Streaming: iterate the PRs in batches (constant memory)

The pattern: open the file, loop over batches of a few light columns, reduce each batch
to a tiny result, and fold it into an accumulator. Memory stays at one batch regardless
of the table size.

In [6]:
merged_per_agent: dict[str, int] = {}
seen = 0
pf = pq.ParquetFile(table_path(PR_TABLE))
for batch in pf.iter_batches(batch_size=BATCH_ROWS, columns=["agent", "merged_at"]):
    df = batch.to_pandas()
    counts = df[df["merged_at"].notna()].groupby("agent").size()
    for agent, n in counts.items():
        merged_per_agent[agent] = merged_per_agent.get(agent, 0) + int(n)
    seen += len(df)
print(f"streamed {seen:,} rows in batches of {BATCH_ROWS:,}")
for agent, n in sorted(merged_per_agent.items(), key=lambda kv: -kv[1]):
    print(f"  {agent:14s} {n:>10,} merged PRs")

streamed 7,685,281 rows in batches of 65,536
  OpenAI_Codex    3,315,477 merged PRs
  Claude_Code     1,378,706 merged PRs
  Copilot         1,106,158 merged PRs
  Google_Jules      187,790 merged PRs
  Cursor            182,263 merged PRs
  Devin              46,398 merged PRs


## 5. Predicate pushdown: read only the rows you need

`filters` is a list of `(column, op, value)` triples that are AND-ed; ops: `=`, `!=`, `<`,
`>`, `<=`, `>=`, `in`, `not in`. Timestamps are ISO-8601 strings
(`"2026-01-15T10:20:30Z"`), so string comparison works for date ranges. Below: Claude Code
PRs in popular repositories created in Q1 2026, light columns only.

In [7]:
prs = pq.read_table(
    table_path("pull_request"),
    columns=LIGHT_PR_COLS,
    filters=[("agent", "=", "Claude_Code"),
             ("created_at", ">=", "2026-01-01"),
             ("created_at", "<", "2026-04-01")],
).to_pandas()
print(f"{len(prs):,} Claude Code PRs in Q1 2026 (popular repos), "
      f"{prs['merged_at'].notna().mean():.1%} merged")
prs.head()

130,016 Claude Code PRs in Q1 2026 (popular repos), 70.5% merged


,id,number,agent,user,state,created_at,closed_at,merged_at,repo_url
0,3960078964,904,Claude_Code,brendan-kellam,closed,2026-02-18T22:32:42Z,2026-02-18T23:09:16Z,2026-02-18T23:09:16Z,https://api.github.com/repos/sourcebot-dev/sou...
1,3960079315,923,Claude_Code,human-dot-bot[bot],closed,2026-02-18T22:32:51Z,2026-02-18T23:09:30Z,2026-02-18T23:09:30Z,https://api.github.com/repos/passportxyz/passp...
2,3960079654,3874,Claude_Code,human-dot-bot[bot],closed,2026-02-18T22:32:56Z,2026-02-19T00:00:21Z,2026-02-19T00:00:21Z,https://api.github.com/repos/passportxyz/passport
3,3960081282,20431,Claude_Code,marcomarandiz,closed,2026-02-18T22:33:30Z,2026-04-03T04:14:52Z,None,https://api.github.com/repos/openclaw/openclaw
4,3960081787,107,Claude_Code,rushikb,closed,2026-02-18T22:33:38Z,2026-03-15T09:21:26Z,None,https://api.github.com/repos/HeroBlackInk/ulti...


## 6. Joining the per-PR sub-resource tables to the PR table

Most sub-resource tables carry `pr_id` = `pull_request.id`. The one exception is
`pr_review_comments`, which carries the API URL of the PR instead; build the same key on
the PR side as `repo_url + "/pulls/" + number`. Below: for Devin's PRs, how many comments /
reviews / review comments each PR received from human users. The sub-resource tables are
filtered down to those PRs *while reading*.

In [8]:
prs = pq.read_table(table_path("pull_request"), columns=["id", "number", "repo_url"],
                    filters=[("agent", "=", "Devin")]).to_pandas()
ids = prs["id"].tolist()

comments = pq.read_table(table_path("pr_comments"), columns=["pr_id", "user_type"],
                         filters=[("pr_id", "in", ids), ("user_type", "=", "User")]).to_pandas()
reviews = pq.read_table(table_path("pr_reviews"), columns=["pr_id", "state", "user_type"],
                        filters=[("pr_id", "in", ids), ("user_type", "=", "User")]).to_pandas()

prs["pull_request_url"] = prs["repo_url"] + "/pulls/" + prs["number"].astype(str)
review_comments = pq.read_table(
    table_path("pr_review_comments"), columns=["pull_request_url", "user_type"],
    filters=[("pull_request_url", "in", prs["pull_request_url"].tolist()),
             ("user_type", "=", "User")]).to_pandas()
review_comments = review_comments.merge(prs[["id", "pull_request_url"]], on="pull_request_url")

per_pr = pd.DataFrame({
    "comments": comments.groupby("pr_id").size(),
    "reviews": reviews.groupby("pr_id").size(),
    "review_comments": review_comments.groupby("id").size(),
}).reindex(ids).fillna(0).astype(int)
print(f"{len(prs):,} Devin PRs; human activity per PR (mean):")
print(per_pr.mean().round(2).to_string())
print("review verdicts:", reviews["state"].value_counts().to_dict())

11,219 Devin PRs; human activity per PR (mean):
comments           0.33
reviews            0.87
review_comments    0.54
review verdicts: {'APPROVED': 4968, 'COMMENTED': 3959, 'CHANGES_REQUESTED': 732, 'DISMISSED': 101}


## 7. The 12 GB table: stream patches of merged PRs and write them out as you go

`pr_commit_details` has one row per (commit, file); `patch` is the diff. Never collect the
hits in a list: write each batch's matches to disk (JSONL here; parquet via
`pq.ParquetWriter` works the same way) and move on.

This cell streams the table **over the network** through `HfFileSystem` (row group by row
group, ~170 MB each) and stops after `MAX_PATCHES` hits, so nothing is downloaded. For a
full pass, download once with `table_path("pr_commit_details")` and stream the local file
the same way (a few minutes).

In [9]:
import json

MAX_PATCHES = 500   # keeps the demo short; drop the limit to process everything

merged = pq.read_table(table_path("pull_request"), columns=["id", "merged_at"]).to_pandas()
merged_ids = set(merged.loc[merged["merged_at"].notna(), "id"])

if DATA_DIR:
    source = table_path("pr_commit_details")
else:
    source = HfFileSystem().open(f"datasets/{HF_REPO}@{HF_REVISION}/pr_commit_details.parquet", "rb")

out_path = OUT_DIR / "sample_patches.jsonl"
n_written = 0
t0 = time.time()
pf = pq.ParquetFile(source)
with out_path.open("w") as out:
    for batch in pf.iter_batches(batch_size=32_768,
                                 columns=["pr_id", "sha", "filename", "status", "patch"]):
        df = batch.to_pandas()
        hits = df[df["pr_id"].isin(merged_ids) & df["patch"].notna()]
        for row in hits.itertuples(index=False):
            out.write(json.dumps(row._asdict()) + "\n")
            n_written += 1
            if n_written >= MAX_PATCHES:
                break
        if n_written >= MAX_PATCHES:
            break
print(f"wrote {n_written:,} patches from merged PRs to {out_path} in {time.time() - t0:.1f} s")
print(json.loads(out_path.open().readline())["patch"][:600])

wrote 500 patches from merged PRs to output/sample_patches.jsonl in 1.1 s
@@ -17,11 +17,12 @@ export const config = { runtime: 'edge' };
 
 const UA = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36';
 const CACHE_TTL_SECONDS = 3600;
+const GITHUB_API_HEADERS = Object.freeze({ Accept: 'application/vnd.github.v3+json', 'User-Agent': UA });
 
 async function fetchGitHubOrg(name) {
   try {
     const res = await fetch(`https://api.github.com/orgs/${encodeURIComponent(name)}`, {
-      headers: { 'Accept': 'application/vnd.github.v3+json', 'User-Agent': UA },
+      headers: GITHUB_API_HEADERS,
    


## Where to go next

* [`dataset_overview.ipynb`](dataset_overview.ipynb): the per-agent summary table and cumulative-PR figure from the dataset card.
* [`language_usage.ipynb`](language_usage.ipynb): which languages each agent works in.
* [`productivity.ipynb`](productivity.ipynb): PR outcomes, turnaround time and acceptance rate over time.

Join keys and sizes of every table: the [table cheat sheet](https://huggingface.co/datasets/hao-li/AIDev/blob/v5/scripts/README.md#table-cheat-sheet).